In [43]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings('ignore')


df = sns.load_dataset("titanic")

In [44]:
df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


In [45]:
# preprocessing the data
df.isnull().sum()

survived         0
pclass           0
sex              0
age            177
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
deck           688
embark_town      2
alive            0
alone            0
dtype: int64

In [46]:
df['age']=df['age'].fillna(df['age'].mean()).astype(int)

df['embarked'].fillna(df['embarked'].mode()[0], inplace=True)

print("Total passengers:", len(df))

survival_summary = pd.crosstab(
    df["sex"],
    df["survived"],
    margins=True,
    margins_name="Total"
).rename(columns={0: "Did not survive", 1: "Survived"})

print(survival_summary)

Total passengers: 891
survived  Did not survive  Survived  Total
sex                                       
female                 81       233    314
male                  468       109    577
Total                 549       342    891


In [47]:
df=pd.get_dummies(df,columns=['sex'],drop_first=True)

df.sample(5)

,survived,pclass,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone,sex_male
510,1,3,29,0,0,7.75,Q,Third,man,True,NaN,Queenstown,yes,True,True
854,0,2,44,1,0,26.00,S,Second,woman,False,NaN,Southampton,no,False,False
814,0,3,30,0,0,8.05,S,Third,man,True,NaN,Southampton,no,True,True
801,1,2,31,1,1,26.25,S,Second,woman,False,NaN,Southampton,yes,False,False
678,0,3,43,1,6,46.90,S,Third,woman,False,NaN,Southampton,no,False,False


In [48]:
df.drop(['deck', 'embark_town', 'alive', 'class', 'who', 'adult_male'], axis=1, inplace=True)
df.head()

,survived,pclass,age,sibsp,parch,fare,embarked,alone,sex_male
0,0,3,22,1,0,7.2500,S,False,True
1,1,1,38,1,0,71.2833,C,False,False
2,1,3,26,0,0,7.9250,S,True,False
3,1,1,35,1,0,53.1000,S,False,False
4,0,3,35,0,0,8.0500,S,True,True


In [49]:
df['alone']=df['alone'].map({True:1,False:0})
df['sex_male']=df['sex_male'].map({True:1,False:0})
df.head()

,survived,pclass,age,sibsp,parch,fare,embarked,alone,sex_male
0,0,3,22,1,0,7.2500,S,0,1
1,1,1,38,1,0,71.2833,C,0,0
2,1,3,26,0,0,7.9250,S,1,0
3,1,1,35,1,0,53.1000,S,0,0
4,0,3,35,0,0,8.0500,S,1,1


In [50]:
from sklearn.preprocessing import OneHotEncoder
import pandas as pd

encoder = OneHotEncoder(sparse_output=False)

encoded = encoder.fit_transform(df[['pclass', 'embarked']]).astype(int)

encoded_df = pd.DataFrame(
    encoded,
    columns=encoder.get_feature_names_out(['pclass', 'embarked']),
    index=df.index
)

df = pd.concat([df, encoded_df], axis=1)
df.drop(['pclass', 'embarked'], axis=1, inplace=True)
df.head()

,survived,age,sibsp,parch,fare,alone,sex_male,pclass_1,pclass_2,pclass_3,embarked_C,embarked_Q,embarked_S
0,0,22,1,0,7.2500,0,1,0,0,1,0,0,1
1,1,38,1,0,71.2833,0,0,1,0,0,1,0,0
2,1,26,0,0,7.9250,1,0,0,0,1,0,0,1
3,1,35,1,0,53.1000,0,0,1,0,0,0,0,1
4,0,35,0,0,8.0500,1,1,0,0,1,0,0,1


In [51]:
from sklearn.preprocessing import StandardScaler

col=['age', 'fare']
scaler = StandardScaler()
df[col] = scaler.fit_transform(df[col])
df.head()

,survived,age,sibsp,parch,fare,alone,sex_male,pclass_1,pclass_2,pclass_3,embarked_C,embarked_Q,embarked_S
0,0,-0.580044,1,0,-0.502445,0,1,0,0,1,0,0,1
1,1,0.650112,1,0,0.786845,0,0,1,0,0,1,0,0
2,1,-0.272505,0,0,-0.488854,1,0,0,0,1,0,0,1
3,1,0.419458,1,0,0.420730,0,0,1,0,0,0,0,1
4,0,0.419458,0,0,-0.486337,1,1,0,0,1,0,0,1


In [52]:
from sklearn.model_selection import train_test_split
X=df.drop('survived',axis=1)
y=df['survived']
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [53]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()
model.fit(X_train, y_train)
y_predict=model.predict(X_test)
print(y_predict)

[0 0 0 1 1 1 1 0 1 1 0 0 0 0 0 1 0 1 0 0 0 0 1 0 0 0 0 0 0 1 0 1 0 1 0 0 0
 1 1 0 0 0 0 0 1 0 0 0 0 0 1 1 0 0 0 1 0 1 1 1 0 1 1 0 0 1 0 0 0 1 1 1 0 1
 0 0 1 1 1 1 0 1 1 0 0 0 1 1 0 0 0 0 0 0 0 0 0 0 1 0 0 0 1 0 0 0 1 0 0 0 1
 0 1 0 1 0 0 0 1 0 0 1 1 0 0 0 1 1 1 0 1 0 0 1 0 1 1 0 0 1 0 1 0 0 0 1 0 0
 1 0 0 0 0 1 0 0 0 1 1 1 0 0 0 1 0 0 0 1 0 0 1 1 0 1 0 0 0 1 1]


In [54]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
accuracy = accuracy_score(y_test, y_predict)
print("Accuracy:", accuracy)
confusion = confusion_matrix(y_test, y_predict)
print("Confusion Matrix:\n", confusion)
classification = classification_report(y_test, y_predict)
print("Classification Report:\n", classification)

Accuracy: 0.7932960893854749
Confusion Matrix:
 [[90 15]
 [22 52]]
Classification Report:
               precision    recall  f1-score   support

           0       0.80      0.86      0.83       105
           1       0.78      0.70      0.74        74

    accuracy                           0.79       179
   macro avg       0.79      0.78      0.78       179
weighted avg       0.79      0.79      0.79       179



In [55]:
# knn model
from sklearn.neighbors import KNeighborsClassifier
knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(X_train, y_train)
y_knn_predict = knn_model.predict(X_test)
print(y_knn_predict)
knn_accuracy = accuracy_score(y_test, y_knn_predict)
print("KNN Accuracy:", knn_accuracy)
knn_confusion = confusion_matrix(y_test, y_knn_predict)
print("KNN Confusion Matrix:\n", knn_confusion)
knn_classification = classification_report(y_test, y_knn_predict)
print("KNN Classification Report:\n", knn_classification)
df.head()

[0 0 0 1 0 1 1 0 1 1 0 0 0 0 0 1 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 1
 1 1 0 0 0 0 0 1 0 0 0 0 0 0 1 0 1 0 1 0 1 1 1 0 1 1 0 0 1 0 0 1 1 1 1 0 1
 0 0 1 1 1 0 0 1 1 0 0 0 1 1 0 0 1 0 0 0 0 0 0 0 1 1 0 0 1 0 0 0 1 0 0 0 1
 0 1 0 1 0 0 0 1 0 0 1 1 1 0 0 1 1 0 0 1 0 0 0 1 0 1 0 0 0 0 1 0 0 0 1 0 0
 1 0 0 0 0 1 0 0 1 1 1 1 0 0 0 1 0 1 0 1 0 0 1 1 0 0 0 0 0 1 1]
KNN Accuracy: 0.8156424581005587
KNN Confusion Matrix:
 [[93 12]
 [21 53]]
KNN Classification Report:
               precision    recall  f1-score   support

           0       0.82      0.89      0.85       105
           1       0.82      0.72      0.76        74

    accuracy                           0.82       179
   macro avg       0.82      0.80      0.81       179
weighted avg       0.82      0.82      0.81       179



,survived,age,sibsp,parch,fare,alone,sex_male,pclass_1,pclass_2,pclass_3,embarked_C,embarked_Q,embarked_S
0,0,-0.580044,1,0,-0.502445,0,1,0,0,1,0,0,1
1,1,0.650112,1,0,0.786845,0,0,1,0,0,1,0,0
2,1,-0.272505,0,0,-0.488854,1,0,0,0,1,0,0,1
3,1,0.419458,1,0,0.420730,0,0,1,0,0,0,0,1
4,0,0.419458,0,0,-0.486337,1,1,0,0,1,0,0,1


In [56]:
new_pass=[[0, 0.27, 0, 0, 0.20, 0, 0, 0, 1, 0, 1, 0]]

new_pass_prediction = knn_model.predict(new_pass)
print("Prediction for new passenger:", new_pass_prediction)
if new_pass_prediction[0] == 1:
    print("The new passenger is predicted to survive.")
else:
    print("The new passenger is predicted not to survive.")

Prediction for new passenger: [1]
The new passenger is predicted to survive.


In [57]:
# Decision Tree Classifier
from sklearn.tree import DecisionTreeClassifier
tree=DecisionTreeClassifier(random_state=42)
tree.fit(X_train,y_train)
y_tree_predict = tree.predict(X_test)
print(y_tree_predict)
tree_accuracy = accuracy_score(y_test, y_tree_predict)
print("Decision Tree Accuracy:", tree_accuracy)
tree_confusion = confusion_matrix(y_test, y_tree_predict)
print("Decision Tree Confusion Matrix:\n", tree_confusion)
tree_classification = classification_report(y_test, y_tree_predict)
print("Decision Tree Classification Report:\n", tree_classification)

[0 0 0 1 1 1 1 0 1 1 0 0 0 0 0 1 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 0 0 1
 1 1 0 0 0 0 0 0 0 0 0 1 0 1 1 0 1 1 1 1 0 1 0 1 1 1 0 0 1 0 0 0 1 1 1 1 1
 0 0 1 1 1 1 0 1 1 0 1 0 1 1 0 0 1 0 0 0 0 0 1 0 1 0 0 0 1 0 0 1 1 0 0 0 1
 0 1 1 0 0 0 0 1 0 0 1 1 1 0 0 1 1 0 0 1 0 0 1 1 0 1 0 0 0 0 1 0 0 0 1 0 1
 1 0 0 1 0 0 0 0 0 1 1 1 0 0 0 1 0 0 0 1 0 0 0 1 1 0 0 0 0 1 1]
Decision Tree Accuracy: 0.7932960893854749
Decision Tree Confusion Matrix:
 [[87 18]
 [19 55]]
Decision Tree Classification Report:
               precision    recall  f1-score   support

           0       0.82      0.83      0.82       105
           1       0.75      0.74      0.75        74

    accuracy                           0.79       179
   macro avg       0.79      0.79      0.79       179
weighted avg       0.79      0.79      0.79       179



In [58]:
tree_prediction = tree.predict(new_pass)
print("Decision Tree Prediction for new passenger:", tree_prediction)
if tree_prediction[0] == 1:
    print("The new passenger is predicted to survive by Decision Tree.")
else:
    print("The new passenger is predicted not to survive by Decision Tree.")

Decision Tree Prediction for new passenger: [0]
The new passenger is predicted not to survive by Decision Tree.


In [59]:
# Random Forest Classifier
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
rf_y_predict = rf.predict(X_test)
print(rf_y_predict)
rf_accuracy = accuracy_score(y_test, rf_y_predict)
print("Random Forest Accuracy:", rf_accuracy)
rf_confusion = confusion_matrix(y_test, rf_y_predict)
print("Random Forest Confusion Matrix:\n", rf_confusion)
rf_classification = classification_report(y_test, rf_y_predict)
print("Random Forest Classification Report:\n", rf_classification,'\n')


rf_prediction = rf.predict(new_pass)
print("Random Forest Prediction for new passenger:", rf_prediction)
if rf_prediction[0] == 1:
    print("The new passenger is predicted to survive by Random Forest.")
else:
    print("The new passenger is predicted not to survive by Random Forest.")

[0 0 0 1 0 1 1 0 1 1 0 0 0 0 0 1 0 1 0 0 0 1 0 0 0 0 0 0 0 1 0 1 1 1 0 0 1
 1 1 0 0 0 0 0 0 0 0 0 0 0 1 1 0 1 0 1 0 1 1 0 0 1 1 0 0 1 0 0 0 1 1 1 1 1
 0 0 1 1 1 1 0 1 1 0 1 0 1 1 0 0 0 0 0 0 0 0 0 0 1 0 0 0 1 0 0 1 1 0 0 1 1
 0 1 1 0 0 1 0 1 0 0 1 1 1 0 0 1 0 0 0 1 0 0 1 1 0 1 0 0 0 0 1 0 0 0 1 0 1
 1 0 0 0 0 1 0 0 0 1 1 1 0 0 0 1 0 0 0 1 0 0 0 1 1 1 0 0 0 1 1]
Random Forest Accuracy: 0.8156424581005587
Random Forest Confusion Matrix:
 [[90 15]
 [18 56]]
Random Forest Classification Report:
               precision    recall  f1-score   support

           0       0.83      0.86      0.85       105
           1       0.79      0.76      0.77        74

    accuracy                           0.82       179
   macro avg       0.81      0.81      0.81       179
weighted avg       0.81      0.82      0.82       179
 

Random Forest Prediction for new passenger: [1]
The new passenger is predicted to survive by Random Forest.


In [60]:
# support vector machine
from sklearn.svm import SVC
svm_model = SVC(kernel='linear', random_state=42)
svm_model.fit(X_train, y_train)
y_svm_predict = svm_model.predict(X_test)
print(y_svm_predict)
svm_accuracy = accuracy_score(y_test, y_svm_predict)
print("SVM Accuracy:", svm_accuracy)
svm_metrics = classification_report(y_test, y_svm_predict)
print("SVM Classification Report:\n", svm_metrics)
svm_classification = classification_report(y_test, y_svm_predict)
print("SVM Classification Report:\n", svm_classification,'\n')


prediction = svm_model.predict(new_pass)
print("SVM Prediction for new passenger:", prediction)
if prediction[0] == 1:
    print("The new passenger is predicted to survive by SVM.")
else:
    print("The new passenger is predicted not to survive by SVM.")


[0 0 0 1 1 1 1 0 1 1 0 0 1 0 0 1 0 1 0 0 0 0 1 0 0 0 0 0 0 1 0 1 0 1 0 0 0
 1 1 0 0 0 0 0 1 0 0 0 0 0 1 1 0 0 0 1 0 1 1 1 0 1 1 0 0 1 0 0 0 1 1 1 0 1
 0 0 1 1 1 1 0 1 1 0 0 0 1 1 0 0 0 0 0 0 0 0 0 0 1 0 0 0 1 0 0 0 1 0 0 0 1
 0 1 0 0 0 0 0 1 1 0 1 1 0 0 0 1 0 1 0 1 0 0 1 0 1 1 0 0 1 0 1 0 0 1 1 0 0
 1 0 0 0 0 1 0 0 0 1 1 1 0 0 0 1 0 0 0 1 0 0 0 1 0 1 0 0 1 1 1]
SVM Accuracy: 0.7877094972067039
SVM Classification Report:
               precision    recall  f1-score   support

           0       0.80      0.85      0.82       105
           1       0.76      0.70      0.73        74

    accuracy                           0.79       179
   macro avg       0.78      0.78      0.78       179
weighted avg       0.79      0.79      0.79       179

SVM Classification Report:
               precision    recall  f1-score   support

           0       0.80      0.85      0.82       105
           1       0.76      0.70      0.73        74

    accuracy                           0.79       179
  

In [61]:
# naive bayes
from sklearn.naive_bayes import GaussianNB

nb_model = GaussianNB()
nb_model.fit(X_train, y_train)
y_nb_predict = nb_model.predict(X_test)
print(y_nb_predict)
nb_accuracy = accuracy_score(y_test, y_nb_predict)
print("Naive Bayes Accuracy:", nb_accuracy)
nb_metrics = classification_report(y_test, y_nb_predict)
print("Naive Bayes Classification Report:\n", nb_metrics)

prediction = nb_model.predict(new_pass)
print("Naive Bayes Prediction for new passenger:", prediction)


[0 0 0 1 1 1 0 0 1 1 1 0 0 0 0 1 1 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 1 0 0 0 0
 0 1 0 0 0 0 0 1 0 0 0 0 1 1 1 0 1 0 1 0 1 1 0 0 1 1 0 0 1 0 0 0 1 1 1 1 1
 0 0 0 1 1 0 0 1 1 0 1 1 1 1 0 0 0 0 0 0 0 0 1 0 1 0 0 0 1 0 0 0 1 0 0 1 1
 0 1 0 1 0 1 1 1 0 0 1 1 0 0 1 1 1 1 0 0 0 0 0 1 1 1 0 0 0 0 1 0 0 0 1 0 0
 1 0 0 0 0 1 0 0 0 0 1 1 0 0 0 1 0 1 0 1 0 0 1 1 0 1 0 0 0 1 1]
Naive Bayes Accuracy: 0.7597765363128491
Naive Bayes Classification Report:
               precision    recall  f1-score   support

           0       0.78      0.82      0.80       105
           1       0.72      0.68      0.70        74

    accuracy                           0.76       179
   macro avg       0.75      0.75      0.75       179
weighted avg       0.76      0.76      0.76       179

Naive Bayes Prediction for new passenger: [1]


In [62]:
# gradient boosting
from sklearn.ensemble import GradientBoostingClassifier
gb_model = GradientBoostingClassifier(random_state=42)
gb_model.fit(X_train, y_train)
y_gb_predict = gb_model.predict(X_test)
print(y_gb_predict)
gb_accuracy = accuracy_score(y_test, y_gb_predict)
print("Gradient Boosting Accuracy:", gb_accuracy)
gb_metrics = classification_report(y_test, y_gb_predict)
print("Gradient Boosting Classification Report:\n", gb_metrics)

prediction = gb_model.predict(new_pass)
print("Gradient Boosting Prediction for new passenger:", prediction)


[0 0 0 1 1 1 1 0 1 1 1 0 0 0 0 1 1 1 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 1 0 0 0
 1 1 0 0 0 0 0 0 0 0 0 0 0 1 1 0 1 0 1 0 1 1 1 0 1 1 0 0 1 0 0 0 1 1 1 1 1
 0 0 1 1 1 1 0 1 1 0 0 0 1 1 0 0 1 0 0 0 0 0 0 0 1 0 0 0 1 0 0 0 1 0 0 0 1
 0 1 0 0 0 0 0 1 0 0 1 1 1 0 0 1 0 0 0 1 0 0 1 0 0 1 0 0 0 0 1 0 0 0 1 0 0
 1 0 0 0 0 0 0 0 0 1 1 1 0 0 0 1 0 0 0 1 0 0 0 1 0 1 0 0 0 1 1]
Gradient Boosting Accuracy: 0.8156424581005587
Gradient Boosting Classification Report:
               precision    recall  f1-score   support

           0       0.81      0.90      0.85       105
           1       0.83      0.70      0.76        74

    accuracy                           0.82       179
   macro avg       0.82      0.80      0.80       179
weighted avg       0.82      0.82      0.81       179

Gradient Boosting Prediction for new passenger: [0]
